# Basket Variance Swaps: Data & Feature Engineering

## Overview

This notebook serves as the pipeline for learning model-free variance swap pricing bounds through deep learning. We follow Neufeld & Sester (2021) where the arbitrage-free price bounds depend continuously on market observables, which allow for neural networks (NN) to train on robust pricing surfaces without the necessity for parametric assumptions as seen in model-based strategies.

## Mathematical Foundation

### Model-Free Pricing Framework

Given market data $(K, \sigma, S_0)$ where $K$ is strike prices, $\sigma$ is bid-ask spreads on options, and $S_0$ are spot prices, the model-independent fair value $y(t)$ of a basket variance swap satisfies Theorem II.2 in Neufeld & Sester (2021):

$$y(t) = \inf_{Q \in \mathcal{M}} \mathbb{E}^Q\left[\left(\sum_{i=1}^{d} w_i r_i(t+1)\right)^2 \mid \mathcal{F}_t\right]$$

where $\mathcal{M}$ is the set of all martingale measures consistent with market prices.

Theorem II.2 (Neufeld & Sester): Under arbitrage-free market conditions, the map $(K, \sigma, S_0) \mapsto [D_{\underline{\mathcal{P}}}(K,\sigma,S_0), D^{\overline{\mathcal{P}}}(K,\sigma,S_0)]$ is continuous, where $D$ denotes price bounds.

Implication: By Cybenko's Universal Approximation Theorem, a feedforward neural network can learn this continuous mapping with arbitrary precision.

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [11]:
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [12]:
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data_dir = project_root / 'output' / 'datasets'
Xtrain = np.load(data_dir / 'features' / 'X_train.npy')
Xval = np.load(data_dir / 'features' / 'X_val.npy')
Xtest = np.load(data_dir / 'features' / 'X_test.npy')
ytrain = np.load(data_dir / 'labels' / 'y_train.npy')
yval = np.load(data_dir / 'labels' / 'y_val.npy')
ytest = np.load(data_dir / 'labels' / 'y_test.npy')
dates_train = np.load(data_dir / 'features' / 'dates_train.npy', allow_pickle=True)
dates_val = np.load(data_dir / 'features' / 'dates_val.npy', allow_pickle=True)
dates_test = np.load(data_dir / 'features' / 'dates_test.npy', allow_pickle=True)

In [33]:
print(f'Train: {Xtrain.shape}, Val: {Xval.shape}, Test: {Xtest.shape}')
print(f'Train dates: {dates_train[0]} to {dates_train[-1]}')
print(f'Val dates: {dates_val[0]} to {dates_val[-1]}')
print(f'Test dates: {dates_test[0]} to {dates_test[-1]}')

Train: (1448, 274), Val: (180, 274), Test: (298, 274)
Train dates: 2018-01-04 to 2023-12-29
Val dates: 2024-01-03 to 2024-09-30
Test dates: 2024-10-01 to 2025-12-30


## Feature Space Architecture

Feature vector $\phi(t) \in \mathbb{R}^{274}$ constructed from market observables via `src/features/build_features.py`:

Block 1: Spot & Forward Prices (42 dims)

Current spot price $S_i(t)$ and risk-neutral forward price $F_i(t) = S_i(t) e^{(r-q)\Delta t}$ for $i=1,\ldots,21$ assets (2 features × 21 assets).

Block 2: Interest Rates & Dividend Yields (42 dims)

3-month risk-free rate and dividend yield for each asset. Enables computation of forward prices and cost-of-carry adjustments (2 features × 21 assets).

Block 3: Implied Volatility (IV) Surface (147 dims)

IV across 7 moneyness strikes: $\sigma(K,t)$ for $K \in \{-20,-10,-5,\text{ATM},+5,+10,+20\}$ bps per asset (7 × 21).

Block 4: Padding (43 dims)

Reserved dimensions for potential feature expansion, though currently set to zero.

In [35]:
feature_blocks = {
    'Spot & Forward': 42,
    'Rates & Yields': 42,
    'IV Surface': 147,
    'Padding': 43
}

block_df = pd.DataFrame({
    'Block': list(feature_blocks.keys()),
    'Dimensions': list(feature_blocks.values())
})

print(block_df.to_string(index=False))
print(f'\nTotal features: {sum(feature_blocks.values())}')

         Block  Dimensions
Spot & Forward          42
Rates & Yields          42
    IV Surface         147
       Padding          43

Total features: 274


## Data Quality Assurance

Time-series split (1 sample: 1 trading day):

- Train: 2018–2023 (1,448 samples)
- Validation: 2024 Jan-Sep (180 samples)
- Test: 2024 Oct–2025 Dec (298 samples)

where all features are standardised with $\mathcal{N}(0,1)$ scaling; no NaN or Inf values.

In [20]:
splits = {'Train': (Xtrain, ytrain, dates_train), 'Val': (Xval, yval, None), 'Test': (Xtest, ytest, dates_test)}

In [21]:
for split_name, (X, y, dates) in splits.items():
    print(f'{split_name}: {X.shape[0]} samples, {X.shape[1]} features')
    print(f'  NaN count: {np.isnan(X).sum()}, Inf count: {np.isinf(X).sum()}')
    print(f'  Target: mean={y.mean():.4f}, std={y.std():.4f}, range=[{y.min():.4f}, {y.max():.4f}]')
    if dates is not None:
        print(f'  Date range: {dates[0]} to {dates[-1]}')
    print()

Train: 1448 samples, 274 features
  NaN count: 0, Inf count: 0
  Target: mean=-62.1452, std=32.1277, range=[-149.9880, 18.8825]
  Date range: 2018-01-04 to 2023-12-29

Val: 180 samples, 274 features
  NaN count: 0, Inf count: 0
  Target: mean=-58.5074, std=28.6290, range=[-106.2882, 12.3445]

Test: 298 samples, 274 features
  NaN count: 0, Inf count: 0
  Target: mean=-87.5006, std=21.4112, range=[-151.2429, -46.5099]
  Date range: 2024-10-01 to 2025-12-30



## Regime-Aware Training & Evaluation

The feature vector itself is regime-agnostic but the NN training incorporates external regime classification based on realised volatility:

Regime Assignment:
- Realised volatility computed on 60-day rolling window from returns 
  $$\sigma_{\text{RV}}(t) = \sqrt{252 \cdot \text{mean}(r_t^2)}$$

- Regimes are assigned by percentile split:
  - Low Vol: $\sigma < 33^{\text{rd}}$ percentile
  - Mid Vol: $33^{\text{rd}} \leq \sigma < 67^{\text{th}}$ percentile
  - High Vol: $\sigma \geq 67^{\text{th}}$ percentile

Training Strategy:
Single NN trained on all regimes simultaneously. The NN learns a unified pricing function across volatility environments and the performance is evaluated separately respective to regime to assess strategy robustness. This will be illustrated in a later notebook.

## Transaction Costs Structure

Round-trip trade costs decompose into:

$$C_{\text{round-trip}} = C_{\text{entry}} + C_{\text{exit}}$$

where each leg includes 1. bid-ask spread, 2. commission, and 3. market impact:

$$C_{\text{entry}} = \frac{\text{spread}}{2} + \text{commission} + \text{slippage}$$

Total cost: 35 basis points round-trip (15 bps: spread + 5 bps: commission + 10 bps: slippage/leg).

In [24]:
costs_config = {
    'Entry Spread': 15.0,
    'Exit Spread': 15.0,
    'Commission': 5.0,
    'Slippage': 10.0
}

In [25]:
entry_cost = costs_config['Entry Spread']/2 + costs_config['Commission'] + costs_config['Slippage']/2
exit_cost = costs_config['Exit Spread']/2 + costs_config['Commission'] + costs_config['Slippage']/2
total_cost = entry_cost + exit_cost

In [26]:
print('Transaction Cost Breakdown (basis points):')
print(f'  Entry cost: {entry_cost:.1f} bps')
print(f'  Exit cost: {exit_cost:.1f} bps')
print(f'  Round-trip total: {total_cost:.1f} bps')

Transaction Cost Breakdown (basis points):
  Entry cost: 17.5 bps
  Exit cost: 17.5 bps
  Round-trip total: 35.0 bps


## Position Sizing: Kelly Criterion

Optimal position size from Kelly Criterion:

$$f^* = \frac{p \cdot b - q}{b}$$

where $p$ = win probability, $q = 1-p$, $b$ = win/loss ratio.

We use fractional Kelly ($f = 0.25 f^*$) for conservative sizing, combined with vol-targeting:

$$\text{Position Size}_t = f^* \cdot \frac{\sigma_{\text{target}}}{\sigma_t}$$

This adjusts for market regime while respecting risk limits.

## Summary

The data pipeline gives:
- 1,926 trading days (2018–2025) across three distinct volatility regimes
- 274 market-observable features grounded in spot/forward prices, interest rates, dividends, and implied volatility surface
- Zero look-ahead bias via time-series cross-validation splits
- Realistic transaction costs (35 bps round-trip) embedded in backtests
- Regime-aware training with separate evaluation by volatility regime, will be discussed further later